# Tahap 2c — Evaluasi sadar prevalensi, threshold, interpretasi, learning curve, dan uji normalisasi emoji
**Runtime: CPU biasa. Perkiraan waktu: 10–20 menit. Tidak ada pelatihan BERT.** Notebook ini membaca prediksi yang sudah tersimpan
(notebook 20) dan melatih ulang hanya model TF-IDF yang ringan. Semua hasil disimpan ke `JURNAL-OUTPUTS/laptop_jurnal/stage2` (file `stage2c_*`).

Bagian: (1) metrik sadar prevalensi, (2) grid threshold diperluas, (3) fitur TF-IDF terpenting per kategori, (4) learning curve, (5) uji normalisasi emoji / token penekanan.

In [ ]:
import os, json, re, random, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
DRIVE='/content/drive/MyDrive'
DATA4=f'{DRIVE}/JURNAL-DATASET'
OUT4=f'{DRIVE}/JURNAL-OUTPUTS/laptop_jurnal'
PREP4=f'{OUT4}/prep'; PRED4=f'{OUT4}/preds'; S2=f'{OUT4}/stage2'
os.makedirs(S2,exist_ok=True)
def P4(fn):
    p=f'{DATA4}/{fn}'
    return p if os.path.exists(p) else f'{DRIVE}/tesis-dataset 3/{fn}'
CATEGORIES=['Intrusive','Avoidance','Arousal','NegCogMood']
SEED=42; random.seed(SEED); np.random.seed(SEED)
TAG='jurnal_final'
print('setup OK')

## 0. Rekonstruksi data (identik dengan notebook 20)

In [ ]:
norm=lambda s: re.sub(r'\s+',' ',str(s).lower().strip())
y_va=np.load(f'{PREP4}/y_va_v3.npy'); y_te=np.load(f'{PREP4}/y_te_v3.npy'); y_gold=np.load(f'{PREP4}/y_gold_v3.npy')
idx_tr=np.load(f'{PREP4}/idx_tr_all_v3.npy'); idx_va=np.load(f'{PREP4}/idx_va_v3.npy'); idx_ho=np.load(f'{PREP4}/idx_ho_v3.npy')
texts_all=np.load(f'{PREP4}/texts_all_v3.npy',allow_pickle=True).tolist()
set_test=set(norm(texts_all[i]) for i in idx_ho); set_val=set(norm(texts_all[i]) for i in idx_va)
mask_tr=[i for i in idx_tr if norm(texts_all[i]) not in set_test|set_val]
df_gold=pd.read_csv(f'{DATA4}/data_ahli_FIXED.csv',encoding='utf-8-sig')
df_nat=pd.read_csv(f'{DATA4}/anotasi_natural_2000.csv',encoding='utf-8-sig')
df_b3=pd.read_csv(f'{DATA4}/gold_batch3_577.csv',encoding='utf-8-sig')
df_b4=pd.read_csv(P4('gold_batch4_600.csv'),encoding='utf-8-sig')
df_b5=pd.read_csv(P4('gold_batch5_600.csv'),encoding='utf-8-sig')
nat_norm=df_nat['Segmen_teks'].map(norm); mask_nat=~nat_norm.isin(set_test)
b3_norm=df_b3['Segmen_teks'].map(norm)
mask_b3=~(b3_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b3_norm.isin(set(nat_norm[mask_nat])))
b4_norm=df_b4['Segmen_teks'].map(norm)
mask_b4=~(b4_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b4_norm.isin(set(nat_norm[mask_nat]))|b4_norm.isin(set(b3_norm[mask_b3])))
b5_norm=df_b5['Segmen_teks'].map(norm)
mask_b5=~(b5_norm.isin(set(norm(texts_all[i]) for i in mask_tr))|b5_norm.isin(set(nat_norm[mask_nat]))|b5_norm.isin(set(b3_norm[mask_b3]))|b5_norm.isin(set(b4_norm[mask_b4])))
tr_raw=df_gold['Segmen_teks'].iloc[mask_tr].tolist()+df_nat['Segmen_teks'][mask_nat].tolist()+df_b3['Segmen_teks'][mask_b3].tolist()+df_b4['Segmen_teks'][mask_b4].tolist()+df_b5['Segmen_teks'][mask_b5].tolist()
va_raw=df_gold['Segmen_teks'].iloc[idx_va].tolist(); te_raw=df_gold['Segmen_teks'].iloc[idx_ho].tolist()
Y_tr=np.vstack([y_gold[mask_tr],df_nat.loc[mask_nat,CATEGORIES].values.astype(float),df_b3.loc[mask_b3,CATEGORIES].values.astype(float),
                df_b4.loc[mask_b4,CATEGORIES].values.astype(float),df_b5.loc[mask_b5,CATEGORIES].values.astype(float)])
assert len(tr_raw)==5313==len(Y_tr), (len(tr_raw),len(Y_tr))
assert len(va_raw)==135 and len(te_raw)==271
print('train',len(tr_raw),'val',len(va_raw),'test',len(te_raw))

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, mean_squared_error, mean_absolute_error, accuracy_score
from scipy.stats import pearsonr
def binar(y): return (np.asarray(y)>=0.5).astype(int)
def tune_thr(Yv,Pv):
    grid=np.round(np.arange(0.30,0.701,0.01),2); thr=[]
    for k in range(4):
        sc=[f1_score(binar(Yv)[:,k],(Pv[:,k]>=t).astype(int),zero_division=0) for t in grid]
        thr.append(float(grid[int(np.argmax(sc))]))
    return thr
def evaluate(Yt,Pt,thr):
    Yb=binar(Yt); Pb=(Pt>=np.array(thr)).astype(int)
    auc=[roc_auc_score(Yb[:,k],Pt[:,k]) for k in range(4)]
    dom_ok=(np.argmax(Pt,1)==np.argmax(np.asarray(Yt),1)).mean()
    return {'F1_weighted':f1_score(Yb,Pb,average='weighted',zero_division=0),'F1_macro':f1_score(Yb,Pb,average='macro',zero_division=0),
            'AUC_mean':float(np.mean(auc)),'MSE':mean_squared_error(Yt,Pt),'MAE':mean_absolute_error(Yt,Pt),
            'Pearson_r':pearsonr(np.asarray(Yt).ravel(),np.asarray(Pt).ravel())[0],'Dominant_acc':dom_ok,
            **{f'AUC_{c}':a for c,a in zip(CATEGORIES,auc)}}
# baseline trivial semua-positif (sanity check)
triv=evaluate(y_te,np.ones_like(y_te,dtype=float),[0.5]*4)
print('Trivial all-positive  F1w=%.4f  F1m=%.4f'%(triv['F1_weighted'],triv['F1_macro']))

## 0b. Muat prediksi test yang sudah ada

In [ ]:
import ast, glob
names={'TFIDF_Ridge':'TFIDF-Ridge','TFIDF_LinearSVR':'TFIDF-LinearSVR','TFIDF_LogReg':'TFIDF-LogReg','TFIDF_SVD_RBFSVR':'TFIDF-SVD-RBFSVR','BERT_only_IndoBERT_':'BERT-only','Hybrid_BERT_SVR':'Hybrid'}
TEST={}
for k,v in names.items():
    f=f'{S2}/test_pred_{k}.npy'
    if os.path.exists(f): TEST[v]=np.load(f)
    else: print('TIDAK ADA:',f,'| ada:',[os.path.basename(x) for x in glob.glob(S2+'/test_pred_*.npy')])
tab20=pd.read_csv(f'{S2}/stage2_baseline_table.csv',index_col=0)
THR={}
for m,v in zip(['TFIDF-Ridge','TFIDF-LinearSVR','TFIDF-LogReg','TFIDF-SVD-RBFSVR','BERT-only (IndoBERT)','Hybrid BERT-SVR'],list(names.values())[:6]):
    if m in tab20.index: THR[v]=[float(x) for x in re.findall(r'\d+\.\d+|\d+',str(tab20.loc[m,'Thresholds']).replace('np.float64',''))][:4]
print(list(TEST), THR)

## 1. Metrik sadar prevalensi (test, 271)
Menjawab: seberapa banyak kinerja yang hanya berasal dari prevalensi label yang tinggi? Acuan peluang: MCC = 0, balanced accuracy = 0,5, kappa = 0, dan PR-AUC acak = prevalensi.

In [ ]:
from sklearn.metrics import matthews_corrcoef, balanced_accuracy_score, cohen_kappa_score, average_precision_score, confusion_matrix
def pa_cat(y,p,t):
    pred=(p>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    prev=y.mean()
    spec=tn/(tn+fp) if tn+fp else np.nan; rec=tp/(tp+fn) if tp+fn else np.nan
    ap=average_precision_score(y,p) if 0<y.sum()<len(y) else np.nan
    return {'prevalence':prev,'MCC':matthews_corrcoef(y,pred) if len(set(pred))>1 else 0.0,'BalAcc':balanced_accuracy_score(y,pred),
            'Kappa':cohen_kappa_score(y,pred) if len(set(pred))>1 else 0.0,'Specificity':spec,'Recall':rec,'AP':ap,'Lift_AP':ap/prev}
def pa_model(Yt,Pt,thr,idx=None):
    Yb=binar(Yt);
    if idx is not None: Yb,Pt=Yb[idx],Pt[idx]
    Yc=np.asarray(Yt) if idx is None else np.asarray(Yt)[idx]
    from scipy.stats import spearmanr
    rows=[]
    for k in range(4):
        r_=pa_cat(Yb[:,k],Pt[:,k],thr[k]); r_['Spearman']=spearmanr(Yc[:,k],Pt[:,k])[0]; rows.append(r_)
    return pd.DataFrame(rows,index=CATEGORIES)
res={}; summ=[]
for m,Pt in TEST.items():
    df=pa_model(y_te,Pt,THR[m]); res[m]=df
    s=df.mean(); s['Model']=m; summ.append(s)
summ.append(pd.Series({'Model':'Trivial all-positive','prevalence':binar(y_te).mean(),'MCC':0.0,'BalAcc':0.5,'Kappa':0.0,'Specificity':0.0,'Recall':1.0,'AP':binar(y_te).mean(),'Lift_AP':1.0,'Spearman':0.0}))
pa=pd.DataFrame(summ).set_index('Model')[['prevalence','MCC','BalAcc','Kappa','Specificity','Recall','AP','Lift_AP','Spearman']]
print('Rata-rata 4 kategori (test 271):'); print(pa.round(4).to_string()); pa.to_csv(f'{S2}/stage2c_prevalence_aware_mean.csv')
print('\nPer kategori, Hybrid:'); print(res['Hybrid'].round(4).to_string())
pd.concat(res,names=['Model','Category']).to_csv(f'{S2}/stage2c_prevalence_aware_per_category.csv')

In [ ]:
# bootstrap CI (2000) untuk MCC & BalAcc rata-rata, dan selisih berpasangan Hybrid - model
rng=np.random.default_rng(42); n=len(y_te); boots=[rng.integers(0,n,n) for _ in range(2000)]
def boot_metric(m,key):
    out=[]
    for i in boots:
        try: out.append(pa_model(y_te,TEST[m],THR[m],i)[key].mean())
        except Exception: pass
    return np.array(out)
rows=[]
for key in ('MCC','BalAcc'):
    bh=boot_metric('Hybrid',key)
    for m in TEST:
        b=boot_metric(m,key)
        L=min(len(b),len(bh)); d=bh[:L]-b[:L]
        rows.append({'Metric':key,'Model':m,'mean':pa.loc[m,key],'CI95_low':np.percentile(b,2.5),'CI95_high':np.percentile(b,97.5),
                     'Δ(Hybrid−model)':pa.loc['Hybrid',key]-pa.loc[m,key],'Δ_CI_low':np.percentile(d,2.5) if m!='Hybrid' else np.nan,'Δ_CI_high':np.percentile(d,97.5) if m!='Hybrid' else np.nan})
ci=pd.DataFrame(rows); print(ci.round(4).to_string(index=False)); ci.to_csv(f'{S2}/stage2c_prevalence_aware_ci.csv',index=False)
print('\nΔ dengan CI yang tidak melewati 0 => selisih signifikan (95%).')

## 2. Grid threshold diperluas (0,05–0,70) vs grid asli (0,30–0,70)
Menjawab kritik bahwa threshold menempel di batas bawah 0,30. Threshold hanya disetel di validasi.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.svm import LinearSVR, SVR
from sklearn.decomposition import TruncatedSVD
from sklearn.multioutput import MultiOutputRegressor
vec=make_union(TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,sublinear_tf=True,max_features=60000),
               TfidfVectorizer(lowercase=True,analyzer='char_wb',ngram_range=(2,5),min_df=3,sublinear_tf=True,max_features=80000))
X_tr=vec.fit_transform(tr_raw); X_va=vec.transform(va_raw); X_te=vec.transform(te_raw)
print('TF-IDF:',X_tr.shape)
clip=lambda P: np.clip(P,0,1)
VAL,TEST_TF={},{}
# (a) Ridge regresi
m=Ridge(alpha=3.0).fit(X_tr,Y_tr); VAL['TFIDF-Ridge']=clip(m.predict(X_va)); TEST_TF['TFIDF-Ridge']=clip(m.predict(X_te))
# (b) Linear SVR per kategori
m=MultiOutputRegressor(LinearSVR(C=0.3,epsilon=0.0,max_iter=5000,random_state=SEED)).fit(X_tr,Y_tr)
VAL['TFIDF-LinearSVR']=clip(m.predict(X_va)); TEST_TF['TFIDF-LinearSVR']=clip(m.predict(X_te))
# (c) Logistic Regression pada label biner (probabilitas)
def fit_lr(Xa,Ya,Xb):
    out=[]
    for k in range(4):
        lr=LogisticRegression(C=2.0,max_iter=2000,class_weight=None).fit(Xa,binar(Ya)[:,k]); out.append(lr.predict_proba(Xb)[:,1])
    return np.stack(out,1)
VAL['TFIDF-LogReg']=fit_lr(X_tr,Y_tr,X_va); TEST_TF['TFIDF-LogReg']=fit_lr(X_tr,Y_tr,X_te)
# (d) TF-IDF -> SVD(300) -> RBF-SVR (analog klasik dari komponen SVR pada hybrid)
svd=TruncatedSVD(300,random_state=SEED).fit(X_tr)
Z_tr,Z_va,Z_te=svd.transform(X_tr),svd.transform(X_va),svd.transform(X_te)
Z_tr/=np.linalg.norm(Z_tr,axis=1,keepdims=True)+1e-9; Z_va/=np.linalg.norm(Z_va,axis=1,keepdims=True)+1e-9; Z_te/=np.linalg.norm(Z_te,axis=1,keepdims=True)+1e-9
pv,pt=[],[]
for k in range(4):
    s=SVR(kernel='rbf',C=1.0,epsilon=0.05,gamma='scale').fit(Z_tr,Y_tr[:,k]); pv.append(s.predict(Z_va)); pt.append(s.predict(Z_te))
VAL['TFIDF-SVD-RBFSVR']=clip(np.stack(pv,1)); TEST_TF['TFIDF-SVD-RBFSVR']=clip(np.stack(pt,1))
print('model TF-IDF dilatih:',list(VAL))

In [ ]:
# prediksi val BERT & Hybrid (jika ada)
for nm,f in (('BERT-only',f'{PRED4}/{TAG}_val_pred_s{SEED}.npy'),('Hybrid',f'{PRED4}/{TAG}_hybrid_val_pred.npy')):
    if os.path.exists(f): VAL[nm]=np.load(f)
    else:
        cand=glob.glob(f'{PRED4}/*val*pred*'); print('TIDAK ADA:',f,'| kandidat:',[os.path.basename(c) for c in cand])
for a,b in (('TFIDF-Ridge','TFIDF-Ridge'),('TFIDF-LinearSVR','TFIDF-LinearSVR'),('TFIDF-LogReg','TFIDF-LogReg'),('TFIDF-SVD-RBFSVR','TFIDF-SVD-RBFSVR')): pass
def tune_grid(Yv,Pv,lo,hi):
    grid=np.round(np.arange(lo,hi+1e-9,0.01),2); thr=[]
    for k in range(4):
        sc=[f1_score(binar(Yv)[:,k],(Pv[:,k]>=t).astype(int),zero_division=0) for t in grid]; thr.append(float(grid[int(np.argmax(sc))]))
    return thr
rows=[]
for m in VAL:
    if m not in TEST: continue
    for lo in (0.30,0.05):
        thr=tune_grid(y_va,VAL[m],lo,0.70)
        vf=f1_score(binar(y_va),(VAL[m]>=np.array(thr)).astype(int),average='weighted',zero_division=0)
        e=evaluate(y_te,TEST[m],thr); p=pa_model(y_te,TEST[m],thr).mean()
        rows.append({'Model':m,'Grid':f'{lo:.2f}-0.70','Thresholds':thr,'Val_F1w':vf,'Test_F1w':e['F1_weighted'],'Test_MCC':p['MCC'],'Test_BalAcc':p['BalAcc'],'Test_Specificity':p['Specificity']})
gt=pd.DataFrame(rows); print(gt.round(4).to_string(index=False)); gt.to_csv(f'{S2}/stage2c_threshold_grid.csv',index=False)

## 3. Fitur TF-IDF terpenting per kategori (interpretasi)
Ridge pada TF-IDF kata (1–2-gram) dilatih ulang hanya untuk interpretasi. Koefisien positif = kata yang menaikkan skor kategori.

In [ ]:
wv=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=5,sublinear_tf=True,max_features=30000)
Xw_tr=wv.fit_transform(tr_raw); Xw_te=wv.transform(te_raw)
rid=Ridge(alpha=3.0).fit(Xw_tr,Y_tr); fn=np.array(wv.get_feature_names_out())
out=[]
for k,c in enumerate(CATEGORIES):
    co=rid.coef_[k]; top=np.argsort(co)[::-1][:15]; bot=np.argsort(co)[:10]
    print(f'\n== {c} ==\n naik : '+', '.join(fn[top])+'\n turun: '+', '.join(fn[bot]))
    for r,i in enumerate(top): out.append({'Category':c,'Direction':'positive','Rank':r+1,'Term':fn[i],'Coef':co[i]})
    for r,i in enumerate(bot): out.append({'Category':c,'Direction':'negative','Rank':r+1,'Term':fn[i],'Coef':co[i]})
pd.DataFrame(out).to_csv(f'{S2}/stage2c_top_terms.csv',index=False)

## 4. Learning curve (Ridge, TF-IDF kata) — apakah lebih banyak data membantu?

In [ ]:
from sklearn.metrics import roc_auc_score
rng=np.random.default_rng(42); Nn=len(tr_raw); rows=[]
for frac in (0.1,0.2,0.4,0.6,0.8,1.0):
    for rep in range(3):
        idx=rng.choice(Nn,int(Nn*frac),replace=False) if frac<1 else np.arange(Nn)
        if frac==1.0 and rep>0: continue
        v=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,sublinear_tf=True,max_features=40000)
        A=v.fit_transform([tr_raw[i] for i in idx]); B=v.transform(te_raw)
        P=np.clip(Ridge(alpha=3.0).fit(A,Y_tr[idx]).predict(B),0,1)
        auc=np.mean([roc_auc_score(binar(y_te)[:,k],P[:,k]) for k in range(4)])
        rows.append({'Fraction':frac,'N':len(idx),'MSE':mean_squared_error(y_te,P),'AUC_mean':auc,'Dominant_acc':(P.argmax(1)==np.asarray(y_te).argmax(1)).mean()})
lc=pd.DataFrame(rows).groupby(['Fraction','N']).mean().reset_index(); print(lc.round(4).to_string(index=False)); lc.to_csv(f'{S2}/stage2c_learning_curve.csv',index=False)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(9,3.2)); ax[0].plot(lc.N,lc.MSE,'o-'); ax[0].set_xlabel('Training texts'); ax[0].set_ylabel('Test MSE'); ax[1].plot(lc.N,lc.AUC_mean,'o-'); ax[1].set_xlabel('Training texts'); ax[1].set_ylabel('Mean AUC-OVR')
plt.tight_layout(); plt.savefig(f'{S2}/stage2c_learning_curve.png',dpi=200); plt.show()

## 5. Uji normalisasi emoji dan token penekanan (proksi untuk "unique-code preprocessing")
TF-IDF kata dilatih pada tiga versi teks: mentah, emoji/penekanan dibuang, dan versi "unique-code" (emoji → token kelompok emosi, tanda baca berulang → token penekanan). Ini hanya bukti pendukung, bukan pengganti ablasi pada BERT.

In [ ]:
import re
EMO=re.compile('[\U0001F300-\U0001FAFF\u2600-\u27BF\u2764]')
NEG=set('😢😭😔😞😟😰😨😱😡😠🤬😤💔😩😫😖😣🥺')
POS=set('😊😀😁😂😄😍❤🙂😃🥰😘👍😆')
REP=re.compile(r'([!?.…])\1{1,}')
def v_raw(t): return t
def v_strip(t): return REP.sub(r'\1',EMO.sub(' ',t))
def v_unique(t):
    t=EMO.sub(lambda m:' emoneg ' if m.group(0) in NEG else (' emopos ' if m.group(0) in POS else ' emolain '),t)
    return REP.sub(' tegaskan ',t)
share_emo=np.mean([bool(EMO.search(t)) for t in tr_raw]); share_rep=np.mean([bool(REP.search(t)) for t in tr_raw])
print(f'Teks latih dengan emoji: {share_emo:.1%}; dengan tanda baca berulang: {share_rep:.1%}')
R={}
for nm,fn_ in (('raw',v_raw),('strip',v_strip),('unique',v_unique)):
    v=TfidfVectorizer(lowercase=True,ngram_range=(1,2),min_df=2,sublinear_tf=True,max_features=40000)
    A=v.fit_transform([fn_(t) for t in tr_raw]); B=v.transform([fn_(t) for t in te_raw])
    R[nm]=np.clip(Ridge(alpha=3.0).fit(A,Y_tr).predict(B),0,1)
rows=[]
rng=np.random.default_rng(42); boots=[rng.integers(0,len(y_te),len(y_te)) for _ in range(5000)]
se=lambda P:((np.asarray(y_te)-P)**2).mean(1)
for nm in R:
    auc=np.mean([roc_auc_score(binar(y_te)[:,k],R[nm][:,k]) for k in range(4)])
    row={'Variant':nm,'MSE':mean_squared_error(y_te,R[nm]),'AUC_mean':auc}
    if nm!='unique':
        d=se(R[nm])-se(R['unique']); bs=[d[i].mean() for i in boots]
        row.update({'ΔMSE (variant − unique)':d.mean(),'CI_low':np.percentile(bs,2.5),'CI_high':np.percentile(bs,97.5)})
    rows.append(row)
em=pd.DataFrame(rows); print(em.round(5).to_string(index=False)); em.to_csv(f'{S2}/stage2c_emoji_test.csv',index=False)
print('\nCI yang melewati 0 => tidak ada bukti bahwa normalisasi membantu (pada TF-IDF).')

In [ ]:
import numpy as np, pandas as pd

yt = np.asarray(y_te)                      # skor emas test (271 x 4)
mu = np.asarray(Y_tr).mean(0)              # rata-rata skor per kategori dari data latih
Pm = np.tile(mu, (len(yt), 1))             # prediktor rata-rata
Yb = binar(y_te)                           # label biner emas (skor >= 0,5)

out = {
    'MeanPredictor_MSE':  ((yt - Pm) ** 2).mean(),
    'MeanPredictor_RMSE': np.sqrt(((yt - Pm) ** 2).mean()),
    'MeanPredictor_MAE':  np.abs(yt - Pm).mean(),
    'AllPositive_ExactMatch': (Yb == 1).all(1).mean(),
    'AllPositive_Hamming':    (Yb != 1).mean(),
}
res = pd.Series(out)
print(res.round(4).to_string())
res.to_csv(f'{S2}/stage2c_trivial_baselines.csv')

## Selesai
Kirim ke saya **screenshot atau file CSV** berikut (folder `stage2`): `stage2c_prevalence_aware_mean.csv`, `stage2c_prevalence_aware_ci.csv`, `stage2c_threshold_grid.csv`, `stage2c_top_terms.csv`, `stage2c_learning_curve.csv` (+ `.png`), `stage2c_emoji_test.csv`.
Kalau ada sel yang error, kirim screenshot error-nya saja.